# RadCluster_2_1 — Digital-Twin Campaign Control (T3 rev 6)

**Run this same notebook, unchanged, on every participating machine.**
It identifies the host itself and takes its machine index, worker count and
row assignment from `machines.json`. Nothing here is machine-specific, and
nothing needs editing — that is the point: the one value whose mistyping
silently corrupts a campaign (`--machine k`) is no longer typed at all.

Run the sections in order. **Sections 0–4 are the pre-flight and must all pass
before section 5 launches the real run.**

| | machine | slots | speed | rows |
|---|---|---|---|---|
| 0 | MacBook Pro | 14 | 1.00 measured | 480 |
| 1 | Matrix-PC | 4 | 0.40 declared | 55 |
| 2 | Nasr Workstation | 14 | 0.50 declared | 240 |
| 3 | MacBook Air | 8 | 0.85 declared | 233 |

Grid is frozen: `I=30000, V=5000`, 1 dpa, `bin_moment`, `active_window`.
Its fingerprint is **`run_cfg_sha = 1eaa3b2c3d9d0c7b`** — section 4 checks it.


## 0 — Setup (no edits needed)


In [ ]:
import json, subprocess, sys, time
from pathlib import Path

HERE    = Path.cwd() if (Path.cwd() / 'run_ensemble.py').exists() else Path('RadCluster_2_1/digital_twin')
sys.path.insert(0, str(HERE))
import campaign_ops as ops
import run_ensemble as RE

DESIGN   = HERE / 'design' / 'T3_rev6.csv'
RESULTS  = HERE / 'results'
REGISTRY = HERE / 'machines.json'
PY       = sys.executable

# The frozen grid. Byte-identical on every machine -- do not edit.
GRID = ['--equations','bin_moment', '--I','30000', '--V','5000',
        '--i-discrete','50', '--v-discrete','5', '--i-bin','25', '--v-bin','25',
        '--i-mobile-default','50', '--v-mobile-default','5',
        '--dose','1.0', '--rtol','1e-6', '--solver-mode','active_window']
EXPECTED_RUN_CFG_SHA = '1eaa3b2c3d9d0c7b'

print('working in', HERE)


## 1 — Pull, so every machine runs the same code

The design, the registry and the worker must match across machines or the
rows cannot be pooled. `merge_and_sobol` compares `git_sha`, `solver_sha256`,
`workbook_sha256`, `design_sha256`, **`run_cfg_sha`**, `weights_sha` and `of`,
and reports a PROVENANCE SPLIT if any of them disagree.


In [ ]:
print(subprocess.run(['git','pull','--ff-only'], cwd=HERE.parent.parent,
                     capture_output=True, text=True).stdout.strip())
print('HEAD =', subprocess.run(['git','rev-parse','--short','HEAD'], cwd=HERE.parent.parent,
                               capture_output=True, text=True).stdout.strip())


## 2 — Who am I?

Detection is by host fingerprint, and it **fails loudly** on no match or an
ambiguous match rather than guessing. A wrong index means two machines compute
the same rows and some rows are computed by nobody — which surfaces at merge
time as *rows MISSING*, indistinguishable from a machine that never reported.

If it refuses, run **`ops_register(k)`** in the next cell with this machine's
index, then commit and push `machines.json`.


In [ ]:
reg   = RE.load_registry(REGISTRY)
facts = RE._host_facts()
print('host:', {k: v for k, v in facts.items() if k != 'env'})
me = RE.detect_machine(reg, facts)      # raises SystemExit if unrecognised

W    = [float(w) for w in reg['weights'].split(',')]
rows, meta = RE.read_design(DESIGN)
mine = [r for r in rows if RE.assign_machine(int(r['row_id']), reg['of'], W) == me['index']]
print(f"\n  machine {me['index']} = {me['name']}")
print(f"  slots {me['slots']}  speed {me['speed']} ({me['speed_source']})  weight {me['weight']}")
print(f"  owns {len(mine)} of {len(rows)} rows")


In [ ]:
# ONLY if section 2 refused to identify this host:
#   RE.register_host(<index>, REGISTRY)   then commit and push machines.json
# Do not guess an index -- check the table at the top of this notebook.


## 3 — Solver build and cross-machine agreement

`check_machine` compares 12 quantities against `machine_reference.json` at
`rtol=1e-6`. It is what makes a *different build* poolable with the others —
each machine compiles its own solver, so the binaries genuinely differ.

> Known gap: the probe runs `equations="discrete"`, while the campaign runs
> `bin_moment`. It verifies the build, not the code path in use.


In [ ]:
info = ops.ensure_solver()          # ops.ensure_solver(force=True) to rebuild
r = subprocess.run([PY, str(HERE / 'check_machine.py')], cwd=HERE,
                   capture_output=True, text=True)
print(r.stdout[-2500:])
assert r.returncode == 0, 'AGREEMENT GATE FAILED — do not contribute rows from this build'


## 4 — PRE-FLIGHT: two real rows, bounded

**Do not skip this.** It is the check whose absence cost two aborted Hoffman2
submissions. Every row there failed in 5–8 s with `d100=nan` while the
provenance line looked perfect — a provenance line prints *before* any row
runs and proves only that the config was assembled.

The suspected cause is unresolved and **may affect any Linux host**: rows
succeed through a direct call but fail through `run_ensemble`'s multiprocessing
pool at the production grid (Linux forks where macOS spawns, and the solver's
stdout is drained on threads). So this must be run *on this machine*.

**Pass = two rows, each taking roughly an hour, `ok`, and the expected
`run_cfg_sha`. Rows that come back in seconds with `nan` are the bug.**


In [ ]:
import re as _re
t0 = time.time()
r = subprocess.run([PY, '-u', str(HERE / 'run_ensemble.py'),
                    '--design', str(DESIGN), '--machine', 'auto', *GRID,
                    '--timeout-s', '12000', '--limit', '2', '--workers', '2',
                    '--out', str(HERE / 'results' / '_preflight.jsonl')],
                   cwd=HERE, capture_output=True, text=True)
out = r.stdout + r.stderr
print(out[-2000:])

sha = _re.search(r'"run_cfg_sha": "([0-9a-f]+)"', out)
sha = sha.group(1) if sha else None
n_fail = out.count('FAIL')
print(f"\n  run_cfg_sha  {sha}  (expected {EXPECTED_RUN_CFG_SHA})")
print(f"  FAIL lines   {n_fail}")
print(f"  elapsed      {time.time()-t0:.0f} s")
assert sha == EXPECTED_RUN_CFG_SHA, 'GRID MISMATCH — this machine is not running the frozen grid'
assert n_fail == 0, 'ROWS FAILED — see above; do NOT launch. Report the error text.'
print('\n  PRE-FLIGHT PASSED')


In [ ]:
# Discard the pre-flight rows: they are real and correct, but they were run
# with --limit/--workers overrides and are not part of this machine's share.
for p in RESULTS.glob('_preflight*'):
    p.unlink()
print('pre-flight artefacts removed')


## 5 — Launch

Detached, so the notebook can be closed. Resumption is the design: rerunning
this cell skips `row_id`s already present and only ever adds.

`--timeout-s` is a **budget, not a cliff** — on expiry the solver is asked to
finalize and its partial trajectory is *kept*, contributing to every rung of
the dose ladder it reached.

> Caveat measured 2026-08-06: the solver runs in **segments**, so `--timeout-s`
> bounds a segment's subprocess, not the whole row. Rows of 16 000–20 500 s
> completed under a 12 000 s setting. Treat it as a floor on row cost, not a cap.


In [ ]:
ops.clear_stop()
LOG = RESULTS / f"worker_machine{me['index']}_rev6.log"
cmd = [PY, '-u', str(HERE / 'run_ensemble.py'),
       '--design', str(DESIGN), '--machine', 'auto', *GRID, '--timeout-s', '12000']
print(' '.join(cmd))
with open(LOG, 'w') as fh:
    proc = subprocess.Popen(cmd, cwd=HERE, stdout=fh, stderr=subprocess.STDOUT,
                            start_new_session=True)
print(f'\n  launched pid {proc.pid}  ->  {LOG.name}')
time.sleep(45)
print(open(LOG).read()[:1500])


## 6 — Monitor


In [ ]:
ops.watch(DESIGN, RESULTS, n_machines=reg['of'], interval=120)   # Ctrl-C to stop watching


In [ ]:
# One-shot snapshot instead of the live loop
st = ops.campaign_status(DESIGN, RESULTS, n_machines=reg['of'])
ops.render_status(st, ops.load_targets())


### 6b — Dose-ladder coverage

Truncation of the size tail is **recorded, not gated** (plan §12(q)), and a
timed-out row is **kept** (§12(s)). What decides comparability is the dose:
screen every row at a *common* dose, because dose moves `d_100` by 5× where the
whole `I=50000 → 200000` grid refinement moved it 0.03 %.


In [ ]:
import collections
cov = collections.Counter()
recs = ops.load_results(RESULTS)
for r in recs.values():
    for rung in (r.get('at_dose') or {}):
        cov[rung] += 1
for k in sorted(cov, key=float):
    print(f'  {k:>4s} dpa : {cov[k]:4d} rows')
print('\n  screen at the highest rung most rows reach (see section 8)')


## 7 — Graceful stop

Sets a sentinel: rows in flight finish and are written, no new rows start.


In [ ]:
ops.request_stop('put the real reason here')


In [ ]:
# Resume after a stop
ops.clear_stop()


## 8 — Pool and report (run on ONE machine, after pushing results)

Push results first — `results/*.jsonl` plus the `*.manifest.json`, which is
what sizes the next campaign from measured throughput.

Run the report **both ways**. If the parameter *ranking* is the same with and
without `--require-converged`, truncation did not buy it anything — that is the
empirical form of the claim the no-gating policy rests on.


In [ ]:
for extra in ([], ['--require-converged']):
    print('='*70); print('  merge_and_sobol', *extra); print('='*70)
    r = subprocess.run([PY, str(HERE / 'merge_and_sobol.py'),
                        '--design', str(DESIGN), '--results', str(RESULTS),
                        '--at-dose', '1.0', *extra],
                       cwd=HERE, capture_output=True, text=True)
    print(r.stdout[-4000:])
